In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, Subset, ConcatDataset
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim
import random

SEED=0
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
class NumpyDataset(Dataset):
    def __init__(self, x_path, y_path):
        self.x = np.load(x_path)
        self.y = np.load(y_path)
    
    def __len__(self):
        return self.x.shape[0]
    
    def __getitem__(self, idx):
        x_full = self.x[idx]
        main_x = torch.tensor(x_full[:11, :, :], dtype=torch.float32)
        time_enc = torch.tensor(x_full[11:12, :, :], dtype=torch.float32)
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        return main_x, time_enc, y

class ConvBlock(nn.Module):
    def __init__(self, in_channels, time_channels, out_channels, kernel_size=3, activation=nn.ReLU()):
        super(ConvBlock, self).__init__()
        padding = kernel_size // 2  # same padding for odd kernel sizes
        self.activation = activation
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size, padding=padding)
        self.time_conv = nn.Conv2d(time_channels, out_channels, kernel_size, padding=padding)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size, padding=padding)
        
    def forward(self, x, time_emb):
        # Resize time embedding if its spatial dimensions differ from x
        if time_emb.shape[-2:] != x.shape[-2:]:
            time_emb = F.interpolate(time_emb, size=x.shape[-2:], mode='bilinear', align_corners=False)
        out_x = self.activation(self.conv1(x))
        out_time = self.activation(self.time_conv(time_emb))
        out = out_x + out_time
        out = self.activation(self.conv2(out))
        return out

class EncoderBlock(nn.Module):
    def __init__(self, in_channels, time_channels, out_channels):
        super(EncoderBlock, self).__init__()
        self.conv_block = ConvBlock(in_channels, time_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
    def forward(self, x, time_emb):
        x_conv = self.conv_block(x, time_emb)
        x_pooled = self.pool(x_conv)
        return x_conv, x_pooled

def crop_tensor(skip, target):
    diffY = skip.size(2) - target.size(2)
    diffX = skip.size(3) - target.size(3)
    return skip[:, :, diffY//2 : skip.size(2) - (diffY - diffY//2), 
                   diffX//2 : skip.size(3) - (diffX - diffX//2)]

class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, time_channels, out_channels):
        super(DecoderBlock, self).__init__()
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.conv_block = ConvBlock(in_channels + skip_channels, time_channels, out_channels)
        
    def forward(self, x, skip, time_emb):
        x = self.upsample(x)
        if x.size(2) != skip.size(2) or x.size(3) != skip.size(3):
            skip = crop_tensor(skip, x)
        x = torch.cat([x, skip], dim=1)
        x = self.conv_block(x, time_emb)
        return x

class ConditionalUNet(nn.Module):
    def __init__(self, input_channels=11, time_channels=1, base_filters=64):
        super(ConditionalUNet, self).__init__()
        # Encoder path
        self.enc1 = EncoderBlock(input_channels, time_channels, base_filters)          # 8 -> 64
        self.enc2 = EncoderBlock(base_filters, time_channels, base_filters * 2)          # 64 -> 128
        self.enc3 = EncoderBlock(base_filters * 2, time_channels, base_filters * 4)      # 128 -> 256
        self.enc4 = EncoderBlock(base_filters * 4, time_channels, base_filters * 8)      # 256 -> 512
        
        self.bottleneck = ConvBlock(base_filters * 8, time_channels, base_filters * 16)  # 512 -> 1024
        
        self.dec1 = DecoderBlock(base_filters * 16, base_filters * 8, time_channels, base_filters * 8)  # (1024+512) -> 512
        self.dec2 = DecoderBlock(base_filters * 8, base_filters * 4, time_channels, base_filters * 4)   # (512+256) -> 256
        self.dec3 = DecoderBlock(base_filters * 4, base_filters * 2, time_channels, base_filters * 2)   # (256+128) -> 128
        self.dec4 = DecoderBlock(base_filters * 2, base_filters, time_channels, base_filters)           # (128+64) -> 64
        
        self.out_conv = nn.Conv2d(base_filters, 1, kernel_size=1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x, time_emb):
        # Encoder
        c1, p1 = self.enc1(x, time_emb)   # c1: (B, 64, 247,247); p1: (B, 64, 123,123)
        c2, p2 = self.enc2(p1, time_emb)    # c2: (B, 128, 123,123); p2: (B, 128, 61,61)
        c3, p3 = self.enc3(p2, time_emb)    # c3: (B, 256, 61,61);  p3: (B, 256, 30,30)
        c4, p4 = self.enc4(p3, time_emb)    # c4: (B, 512, 30,30);  p4: (B, 512, 15,15)
        
        bn = self.bottleneck(p4, time_emb)  # bn: (B, 1024, 15,15)
        
        # Decoder
        d1 = self.dec1(bn, c4, time_emb)    # d1: (B, 512, 30,30)
        d2 = self.dec2(d1, c3, time_emb)     # d2: (B, 256, 60,60)
        d3 = self.dec3(d2, c2, time_emb)     # d3: (B, 128, 120,120)
        d4 = self.dec4(d3, c1, time_emb)     # d4: (B, 64, 240,240)
        
        out = self.out_conv(d4)             # (B, 1, 240,240)
        out = self.sigmoid(out)
        # Resize the output to match the original input size (e.g., 247x247)
        out = F.interpolate(out, size=x.shape[-2:], mode='bilinear', align_corners=False)
        return out

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10, device='cuda', model_save_path="best_model.pth"):
    model.to(device)
    best_val_loss = float('inf')
    start_time = time.time()
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for main_x, time_enc, y in train_loader:
            main_x, time_enc, y = main_x.to(device), time_enc.to(device), y.to(device)
            optimizer.zero_grad()
            outputs = model(main_x, time_enc)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * main_x.size(0)
        epoch_loss = running_loss / len(train_loader.dataset)
        
        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for main_x, time_enc, y in val_loader:
                main_x, time_enc, y = main_x.to(device), time_enc.to(device), y.to(device)
                outputs = model(main_x, time_enc)
                loss = criterion(outputs, y)
                val_loss += loss.item() * main_x.size(0)
        val_loss /= len(val_loader.dataset)
        
        print(f"Epoch {epoch+1}/{num_epochs}: Train Loss: {epoch_loss:.4f}, Val Loss: {val_loss:.4f}")
        
        # Save the best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), model_save_path)
    
    total_time = time.time() - start_time
    print(f"Total training time: {total_time:.2f} seconds")
    return model

def compute_ssim_scores(model, dataloader, device='cpu'):
    model.eval()
    ssim_scores = []
    with torch.no_grad():
        for main_x, time_enc, y in dataloader:
            main_x, time_enc, y = main_x.to(device), time_enc.to(device), y.to(device)
            preds = model(main_x, time_enc)
            # Process each sample in the batch individually.
            for i in range(preds.shape[0]):
                pred_img = preds[i, 0, :, :].cpu().numpy()
                gt_img = y[i, 0, :, :].cpu().numpy()
                score = ssim(gt_img, pred_img, data_range=1.0)
                ssim_scores.append(score)
    return ssim_scores


In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for i in range(1,6):
    for j in range((i+1),7):
        for k in range((j+1),8):
            for l in range((k+1),9):
                torch.manual_seed(SEED)
                
                train_1= torch.load(f'training_data/train_dataset_{i}.pt', weights_only=False)
                val_1  = torch.load(f'training_data/val_dataset_{i}.pt', weights_only=False)
                test_1 = torch.load(f'training_data/test_dataset_{i}.pt', weights_only=False)
        
                train_2= torch.load(f'training_data/train_dataset_{j}.pt', weights_only=False)
                val_2  = torch.load(f'training_data/val_dataset_{j}.pt', weights_only=False)
                test_2 = torch.load(f'training_data/test_dataset_{j}.pt', weights_only=False)
    
                train_3= torch.load(f'training_data/train_dataset_{k}.pt', weights_only=False)
                val_3  = torch.load(f'training_data/val_dataset_{k}.pt', weights_only=False)
                test_3 = torch.load(f'training_data/test_dataset_{k}.pt', weights_only=False)

                train_4= torch.load(f'training_data/train_dataset_{l}.pt', weights_only=False)
                val_4  = torch.load(f'training_data/val_dataset_{l}.pt', weights_only=False)
                test_4 = torch.load(f'training_data/test_dataset_{l}.pt', weights_only=False)
        
                train_dataset = ConcatDataset([train_1, train_2, train_3, train_4])
                val_dataset = ConcatDataset([val_1, val_2, val_3, val_4])
                test_dataset = ConcatDataset([test_1, test_2, test_3, test_4])
                
                print(f"Dataset sizes: Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")
                batch_size = 8
                train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
                val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
                test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
                
                model = ConditionalUNet(input_channels=11, time_channels=1, base_filters=64)
                criterion = nn.MSELoss()
                optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
            
                num_epochs = 100
                model = train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs, device,
                                    model_save_path=f"best_model_core4/best_model_{i}_{j}_{k}_{l}.pth")
        
                torch.cuda.empty_cache()